In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# laoding the pre processed data for feature extraction
df_train = pd.read_csv('/content/drive/MyDrive/Week-4/train_processed.csv')
df_test = pd.read_csv('/content/drive/MyDrive/Week-4/test_processed.csv')


In [5]:
df_train['Date'] = pd.to_datetime(df_train['Date'])
df_test['Date'] = pd.to_datetime(df_test['Date'])

In [ ]:
# df_sampled = df.sample(100, random_state=42)

In [6]:
def to_holiday(df, holiday_list):
    # Ensure holiday_list is in datetime format
    holiday_list = pd.to_datetime(holiday_list)
    result = []

    for date in df['Date']:
        # Calculate the timedelta to each holiday
        future_holidays = holiday_list[holiday_list > date]

        # Find the minimum timedelta in days
        if len(future_holidays) > 0:
            min_days = (future_holidays - date).days.min()
        else:
            min_days = 300  # Arbitrary large value if no future holiday

        result.append(min_days)
    return np.array(result)

def after_holiday(df, holiday_list):
    # Ensure holiday_list is in datetime format
    holiday_list = pd.to_datetime(holiday_list)
    result = []

    for date in df['Date']:
        # Calculate the timedelta to each holiday
        future_holidays = holiday_list[holiday_list < date]

        # Find the minimum timedelta in days
        if len(future_holidays) > 0:
            min_days = (date - future_holidays).days.min()
        else:
            min_days = 300  # Arbitrary large value if no future holiday

        result.append(min_days)
    return np.array(result)


In [9]:
def create_columns(df):
    # changing to date time datatype
    df_train['Date'] = pd.to_datetime(df_train['Date'])

    df_train['IsWeekday'] = df_train['Date'].dt.weekday < 5  # True for Monday-Friday
    df_train['IsWeekend'] = ~df_train['IsWeekday']          # True for Saturday-Sunday

    holiday_a = df[df['StateHoliday_a']== 1]['Date'].unique()


    from_holiday_a = to_holiday(df, holiday_a)


    after_holiday_a = after_holiday(df, holiday_a)


    df['Days from Holiday_a'] = from_holiday_a


    df['Days after Holiday_a'] = after_holiday_a


    return df


In [10]:
train_featured = create_columns(df_train)
test_featured = create_columns(df_test)

In [11]:
train_featured.to_csv("feature_train", index = False)
test_featured.to_csv('feature_test', index = False)